# Bloom Filters: The Maybe-Membership Machine

A Bloom filter is a probabilistic data structure. It answers one question quickly: **have we maybe seen this item before?**

Burton Bloom introduced the structure in 1970 as a compact way to trade a small false-positive rate for major memory savings. Bloom filters now appear in databases, caches, web crawlers, blockchains, spell systems, and distributed services.

It can say:

- **definitely not**: at least one required bit is off
- **maybe yes**: all required bits are on

The tradeoff: Bloom filters are tiny and fast, but they can have false positives.

<details>
<summary>Big idea</summary>

Instead of storing every item, store a small row of bits. Each item flips a few bit positions using hash functions. Later, checking the same item looks at those same positions.

</details>

## 1. The Mental Model

A Bloom filter uses three ingredients:

- a fixed-size bit array
- several hash functions
- a rule: all hashed bit positions must be `1` for a maybe-match

Adding an item never stores the item itself. It only turns on bits.

<details>
<summary>Why false positives happen</summary>

Different items can light up overlapping bit positions. An item that was never added may still check positions that are already all `1` because of other items.

</details>

## 2. Build the Objects

Implementation plan:

1. `CatalogItem` names one item we may add or check.
2. `HashProbe` records one hash function landing on one bit position.
3. `FilterStep` stores a snapshot before and after an add or check.
4. `BloomFilter` owns the bit array and membership logic.
5. `BloomReplay` prints the snapshots.

<details>
<summary>Implementation hint</summary>

Python's built-in `hash()` is intentionally randomized between sessions. We will use `hashlib.sha256` so the notebook output stays stable.

</details>

**Object model.** Define `CatalogItem`, `HashProbe`, the named objects used by the next examples.


In [ ]:
from dataclasses import dataclass

import hashlib

import math

@dataclass(frozen=True)
class CatalogItem:
    name: str

    def key(self) -> str:
        return self.name.lower().strip()

    def __str__(self) -> str:
        return self.name

@dataclass(frozen=True)
class HashProbe:
    hash_number: int
    position: int

    def __str__(self) -> str:
        return f"h{self.hash_number}->{self.position}"


**Trace model.** Define `FilterStep`, the structure used to capture replayable algorithm state.


In [ ]:
@dataclass
class FilterStep:
    action: str
    item: CatalogItem
    probes: list[HashProbe]
    bits_before: tuple[int, ...]
    bits_after: tuple[int, ...]
    result: bool | None
    note: str


**Object model.** Define `BloomFilter`, the named objects used by the next examples.


In [ ]:
class BloomFilter:
    def __init__(self, size: int, hash_count: int):
        if size <= 0:
            raise ValueError("Bloom filter size must be positive.")
        if hash_count <= 0:
            raise ValueError("Hash count must be positive.")

        self.size = size
        self.hash_count = hash_count
        self.bits = [0] * size
        self.steps: list[FilterStep] = []

    def _position(self, item: CatalogItem, hash_number: int) -> int:
        text = f"{hash_number}:{item.key()}".encode("utf-8")
        digest = hashlib.sha256(text).hexdigest()
        return int(digest, 16) % self.size

    def probes_for(self, item: CatalogItem) -> list[HashProbe]:
        return [
            HashProbe(hash_number=hash_number, position=self._position(item, hash_number))
            for hash_number in range(1, self.hash_count + 1)
        ]

    def add(self, item: CatalogItem) -> FilterStep:
        probes = self.probes_for(item)
        bits_before = tuple(self.bits)

        for probe in probes:
            self.bits[probe.position] = 1

        step = FilterStep(
            action="add",
            item=item,
            probes=probes,
            bits_before=bits_before,
            bits_after=tuple(self.bits),
            result=None,
            note="Turn on every bit touched by this item.",
        )
        self.steps.append(step)
        return step

    def might_contain(self, item: CatalogItem) -> FilterStep:
        probes = self.probes_for(item)
        bits_before = tuple(self.bits)
        result = all(self.bits[probe.position] == 1 for probe in probes)
        note = "maybe present" if result else "definitely not present"

        step = FilterStep(
            action="check",
            item=item,
            probes=probes,
            bits_before=bits_before,
            bits_after=bits_before,
            result=result,
            note=note,
        )
        self.steps.append(step)
        return step

    def bit_string(self) -> str:
        return "".join(str(bit) for bit in self.bits)

    def false_positive_rate_estimate(self, inserted_count: int) -> float:
        return (1 - math.exp(-self.hash_count * inserted_count / self.size)) ** self.hash_count


## 3. Fill a Tiny Snack Passport

Imagine a festival scanner that tracks which snack names have already been sampled. To save memory, it stores only bits, not the full snack list.

<details>
<summary>Why this is probabilistic</summary>

The filter can prove an item is new when one of its bits is still `0`. But when all bits are `1`, it cannot prove which item originally turned them on.

</details>

In [2]:
snack_filter = BloomFilter(size=18, hash_count=3)
sampled_snacks = [
    CatalogItem("Mochi"),
    CatalogItem("Ramen"),
    CatalogItem("Churro"),
    CatalogItem("Pretzel"),
    CatalogItem("Gelato"),
    CatalogItem("Falafel"),
]

for item in sampled_snacks:
    step = snack_filter.add(item)
    probe_text = ", ".join(str(probe) for probe in step.probes)
    print(f"add {item.name:<8} probes: {probe_text:<22} bits: {snack_filter.bit_string()}")

estimated_rate = snack_filter.false_positive_rate_estimate(inserted_count=len(sampled_snacks))
print(f"\nEstimated false-positive rate: {estimated_rate:.1%}")

add Mochi    probes: h1->9, h2->16, h3->3   bits: 000100000100000010
add Ramen    probes: h1->11, h2->15, h3->9  bits: 000100000101000110
add Churro   probes: h1->4, h2->17, h3->12  bits: 000110000101100111
add Pretzel  probes: h1->1, h2->5, h3->8    bits: 010111001101100111
add Gelato   probes: h1->11, h2->14, h3->5  bits: 010111001101101111
add Falafel  probes: h1->6, h2->1, h3->15   bits: 010111101101101111

Estimated false-positive rate: 25.3%


## 4. Ask Membership Questions

Now we check known and unknown snacks. A Bloom filter should never say **definitely not** for something we already added.

<details>
<summary>Key guarantee</summary>

With normal Bloom filters, there are no false negatives after insertion. Added items always check as maybe present, because their bits were turned on during `add`.

</details>

In [3]:
actual_sampled = {item.key() for item in sampled_snacks}
questions = [
    CatalogItem("Mochi"),
    CatalogItem("Bao"),
    CatalogItem("Gelato"),
    CatalogItem("Taco"),
    CatalogItem("Pretzel"),
    CatalogItem("Noodles"),
    CatalogItem("Churro"),
]

for item in questions:
    step = snack_filter.might_contain(item)
    truth = "actually added" if item.key() in actual_sampled else "not added"
    answer = "maybe yes" if step.result else "definitely not"
    probe_text = ", ".join(str(probe) for probe in step.probes)
    print(f"{item.name:<8} -> {answer:<14} ({truth:<14}) probes: {probe_text}")

Mochi    -> maybe yes      (actually added) probes: h1->9, h2->16, h3->3
Bao      -> definitely not (not added     ) probes: h1->7, h2->10, h3->2
Gelato   -> maybe yes      (actually added) probes: h1->11, h2->14, h3->5
Taco     -> definitely not (not added     ) probes: h1->6, h2->3, h3->10
Pretzel  -> maybe yes      (actually added) probes: h1->1, h2->5, h3->8
Noodles  -> definitely not (not added     ) probes: h1->15, h2->13, h3->15
Churro   -> maybe yes      (actually added) probes: h1->4, h2->17, h3->12


## 5. Replay the Bit Changes

The filter feels more concrete when you can see which bits each item touches.

<details>
<summary>Reading the replay</summary>

For `add`, the bit string can change. For `check`, the bit string stays the same; the filter only reads positions.

</details>

In [4]:
class BloomReplay:
    def __init__(self, steps: list[FilterStep]):
        self.steps = steps

    def show(self, limit: int = 8) -> None:
        for step_number, step in enumerate(self.steps[:limit], start=1):
            probe_text = ", ".join(str(probe) for probe in step.probes)
            print(f"Step {step_number}: {step.action.upper()} {step.item}")
            print(f"  probes: {probe_text}")
            print(f"  before: {''.join(str(bit) for bit in step.bits_before)}")
            print(f"  after:  {''.join(str(bit) for bit in step.bits_after)}")
            if step.result is not None:
                answer = "maybe yes" if step.result else "definitely not"
                print(f"  answer: {answer}")
            print(f"  note: {step.note}\n")


BloomReplay(snack_filter.steps).show(limit=8)

Step 1: ADD Mochi
  probes: h1->9, h2->16, h3->3
  before: 000000000000000000
  after:  000100000100000010
  note: Turn on every bit touched by this item.

Step 2: ADD Ramen
  probes: h1->11, h2->15, h3->9
  before: 000100000100000010
  after:  000100000101000110
  note: Turn on every bit touched by this item.

Step 3: ADD Churro
  probes: h1->4, h2->17, h3->12
  before: 000100000101000110
  after:  000110000101100111
  note: Turn on every bit touched by this item.

Step 4: ADD Pretzel
  probes: h1->1, h2->5, h3->8
  before: 000110000101100111
  after:  010111001101100111
  note: Turn on every bit touched by this item.

Step 5: ADD Gelato
  probes: h1->11, h2->14, h3->5
  before: 010111001101100111
  after:  010111001101101111
  note: Turn on every bit touched by this item.

Step 6: ADD Falafel
  probes: h1->6, h2->1, h3->15
  before: 010111001101101111
  after:  010111101101101111
  note: Turn on every bit touched by this item.

Step 7: CHECK Mochi
  probes: h1->9, h2->16, h3->3
  bef

## 6. Experiments

Bloom filters are controlled by size and hash count. A small crowded filter saves memory but creates more false positives.

<details>
<summary>Experiment hint</summary>

A larger bit array usually lowers the false-positive rate because unrelated items collide less often.

</details>

**Run the experiment.** Advance the algorithm and collect the state changes that make the behavior visible.


In [ ]:
candidate_names = [
    "Bao", "Taco", "Noodles", "Samosa", "Poutine", "Curry", "Bibimbap", "Paella",
    "Pierogi", "Dumpling", "Sushi", "Toast", "Biryani", "Tempura", "Arepa", "Pancake",
    "Satay", "Tamale", "Cannoli", "Baklava", "Udon", "Kimchi", "Gnocchi", "Crepe",
]

false_positives = []

true_negatives = []

for name in candidate_names:
    item = CatalogItem(name)
    if item.key() in actual_sampled:
        continue

    step = snack_filter.might_contain(item)
    if step.result:
        false_positives.append((item, step.probes))
    else:
        true_negatives.append(item)

print("False positives found:")

for item, probes in false_positives[:5]:
    probe_text = ", ".join(str(probe) for probe in probes)
    print(f"  {item.name:<9} looked present because {probe_text} were already 1")

print(f"\nTrue negatives: {len(true_negatives)}")


**Inspect the result.** Evaluate the expression and read the output before changing parameters.


In [ ]:
print(f"False positives: {len(false_positives)}")


In [6]:
def build_filter(size: int, hash_count: int, items: list[CatalogItem]) -> BloomFilter:
    bloom_filter = BloomFilter(size=size, hash_count=hash_count)
    for item in items:
        bloom_filter.add(item)
    return bloom_filter


def count_false_positives(bloom_filter: BloomFilter, names: list[str], actual_keys: set[str]) -> int:
    total = 0
    for name in names:
        item = CatalogItem(name)
        if item.key() not in actual_keys and bloom_filter.might_contain(item).result:
            total += 1
    return total


for size in [12, 18, 32, 64]:
    experiment_filter = build_filter(size=size, hash_count=3, items=sampled_snacks)
    false_positive_count = count_false_positives(experiment_filter, candidate_names, actual_sampled)
    estimated = experiment_filter.false_positive_rate_estimate(len(sampled_snacks))
    print(
        f"size={size:<2} bits_on={sum(experiment_filter.bits):<2}/{size:<2} "
        f"false_positives={false_positive_count:<2} estimated_rate={estimated:.1%}"
    )

size=12 bits_on=10/12 false_positives=16 estimated_rate=46.9%
size=18 bits_on=13/18 false_positives=6  estimated_rate=25.3%
size=32 bits_on=14/32 false_positives=1  estimated_rate=8.0%
size=64 bits_on=16/64 false_positives=0  estimated_rate=1.5%


## What You Should Remember

Bloom filters are useful when memory is tight and a **maybe** answer is acceptable.

- Added items should never become false negatives.
- Unknown items can become false positives.
- More bits usually means fewer collisions.
- More hash functions help up to a point, then they crowd the filter.

<details>
<summary>Where this shows up</summary>

Bloom filters are used in caches, databases, spell checkers, network systems, and large-scale duplicate detection.

</details>

## Visual Trace + Rigor Studio

**Problem frame.** Trade exact membership for compact probabilistic memory.

**Interactive animation target.** Animate hash functions setting bits and queries checking all required positions.

**Correctness handle.** Inserted items never become false negatives as long as bits are never cleared.

**Complexity handle.** O(k) insert and lookup for k hash functions.

**Failure mode to test.** False positives rise as the bit array saturates.

**Studio task.** Keep the item count fixed, shrink the array, and measure how the false-positive rate changes.


In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "courseware").exists():
        sys.path.insert(0, str(candidate))
        break

from courseware import AlgorithmPlayer, AlgorithmTrace, TraceStep, render_trace_table

# Convert the implementation above into snapshots:
# trace = AlgorithmTrace("Topic trace")
# trace.append("start", {"your_state": ...}, "What changed?", invariant="What remains true?")
# AlgorithmPlayer(trace, your_renderer).display()
print("Use AlgorithmTrace to turn this notebook's algorithm into a step-by-step visual player.")
